# SIC 2026 · Inteligencia Artificial
## Clase 7: Precision, recall y F1 con números, y aprendizaje no supervisado

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

La primera parte de la clase es con Nicolaj, trabajando en los proyectos. Esta segunda parte tiene dos momentos:

1. **Aclaración: precision, recall y F1.** En la Clase 6 conversamos sobre la curva precision-recall y la explicación quedó incompleta. La retomamos con números calculados a mano para que no quede ninguna duda.
2. **Aprendizaje no supervisado (Capítulo 6):** K-medias, clustering jerárquico, DBSCAN y PCA, con ejercicios de aplicación.

| Hora | Actividad |
|---|---|
| 20:00 a 20:25 | Parte 1. Precision, recall y F1 |
| 20:25 a 20:35 | Parte 2. ¿Qué es el aprendizaje no supervisado? |
| 20:35 a 21:05 | Parte 3. K-medias: segmentar clientes |
| 21:05 a 21:20 | Parte 4. Clustering jerárquico |
| 21:20 a 21:35 | Parte 5. DBSCAN |
| 21:35 a 21:55 | Parte 6. PCA |
| 21:55 a 22:00 | Cierre |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    """Imprime si un ejercicio está bien resuelto."""
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

print("Listo.")

---
# Parte 1. Precision, recall y F1 con números

## Primero, una corrección

En la Clase 6 dijimos, correctamente, que **cuando sube el recall, baja la precision**. Pero la explicación de *por qué* quedó mal planteada: se dijo que ocurre porque el recall es más exigente con la realidad. No es así. Las dos métricas no se diferencian en cuán exigentes son, sino en **contra qué comparan los aciertos**. Y lo que produce el intercambio entre ellas es **el umbral**. Vamos a verlo con números.

## 1.1 Las dos fórmulas: todo está en el denominador

Recordemos la matriz de confusión:

|  | El modelo dice 0 | El modelo dice 1 |
|---|---|---|
| **En realidad es 0** | Verdadero negativo (VN) | Falso positivo (FP) |
| **En realidad es 1** | Falso negativo (FN) | Verdadero positivo (VP) |

$$\text{Precision} = \frac{VP}{VP + FP} \qquad\qquad \text{Recall} = \frac{VP}{VP + FN}$$

Ambas tienen el mismo numerador: los aciertos positivos (VP). Lo que cambia es el denominador:

| Métrica | Denominador | Pregunta que responde | Está anclada en |
|---|---|---|---|
| **Precision** | VP + FP = todo lo que **el modelo marcó** como positivo | De los que marqué, ¿cuántos eran realmente positivos? | **Las predicciones del modelo** |
| **Recall** | VP + FN = todo lo que **realmente es** positivo | De los positivos reales, ¿cuántos encontré? | **La realidad** |

Una consecuencia importante: **el denominador del recall no depende del modelo**. Si hay 6 positivos reales, siempre son 6, cualquiera sea el umbral. El denominador de la precision, en cambio, cambia cada vez que el modelo marca más o menos casos.

## 1.2 Un ejemplo para calcular a mano

Un modelo estimó la probabilidad de que 12 clientes se vayan de la compañía. Sabemos quiénes se fueron realmente (1) y quiénes no (0). Están ordenados de mayor a menor probabilidad.

In [ ]:
ejemplo = pd.DataFrame({
    "cliente": [f"C{i:02d}" for i in range(1, 13)],
    "probabilidad": [0.95, 0.90, 0.80, 0.75, 0.65, 0.55, 0.45, 0.40, 0.30, 0.20, 0.15, 0.05],
    "real":         [1,    1,    0,    1,    1,    0,    1,    0,    0,    1,    0,    0],
})
print("Positivos reales (se fueron):", ejemplo["real"].sum())
ejemplo

Hay **6 positivos reales**. Ese es el denominador del recall en todos los cálculos que siguen.

### Umbral 0,7

El modelo marca como "se va" a quienes tienen probabilidad mayor o igual a 0,7: **C01, C02, C03 y C04** (4 clientes).

- De esos 4, se fueron realmente C01, C02 y C04: **VP = 3**. C03 no se fue: **FP = 1**.
- De los 6 que se fueron, el modelo no marcó a C05, C07 y C10: **FN = 3**.

$$\text{Precision} = \frac{3}{3 + 1} = 0{,}75 \qquad \text{Recall} = \frac{3}{3 + 3} = 0{,}50$$

### Umbral 0,2

Ahora el modelo marca a todos los que tienen 0,2 o más: **C01 a C10** (10 clientes).

- Aciertos: C01, C02, C04, C05, C07 y C10: **VP = 6**. Falsas alarmas: C03, C06, C08 y C09: **FP = 4**.
- No se le escapa nadie: **FN = 0**.

$$\text{Precision} = \frac{6}{6 + 4} = 0{,}60 \qquad \text{Recall} = \frac{6}{6 + 0} = 1{,}00$$

### Qué pasó al bajar el umbral de 0,7 a 0,2

- El modelo **marcó más clientes** (de 4 a 10). Entre los nuevos marcados había 3 que sí se iban y 3 que no.
- **Recall:** el denominador quedó fijo en 6 y los aciertos subieron de 3 a 6. El recall **sube** de 0,50 a 1,00.
- **Precision:** el denominador creció de 4 a 10, pero los aciertos crecieron menos (de 3 a 6). La precision **baja** de 0,75 a 0,60.

> **El intercambio lo produce el umbral.** Bajarlo hace que el modelo marque más casos: encuentra más positivos reales (sube el recall) pero también comete más falsas alarmas (tiende a bajar la precision). Subirlo hace lo contrario.

## 1.3 Todos los umbrales a la vez

Calculamos lo mismo para cada umbral posible con una función. Usamos los nombres del Capítulo 5: VP, FP y FN.

In [ ]:
def metricas_umbral(prob, real, umbral):
    marcados = prob >= umbral
    vp = int((marcados & (real == 1)).sum())
    fp = int((marcados & (real == 0)).sum())
    fn = int((~marcados & (real == 1)).sum())
    precision = vp / (vp + fp) if vp + fp > 0 else 0.0
    recall = vp / (vp + fn)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {"Umbral": umbral, "Marcados": int(marcados.sum()), "VP": vp, "FP": fp, "FN": fn,
            "Precision": precision, "Recall": recall, "Promedio simple": (precision + recall) / 2, "F1": f1}

prob, real = ejemplo["probabilidad"].values, ejemplo["real"].values
tabla = pd.DataFrame([metricas_umbral(prob, real, u) for u in prob]).round(3)
tabla

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
ejes[0].plot(tabla["Umbral"], tabla["Precision"], marker="o", label="Precision")
ejes[0].plot(tabla["Umbral"], tabla["Recall"], marker="o", label="Recall")
ejes[0].set_xlabel("Umbral"); ejes[0].set_title("Al bajar el umbral (hacia la izquierda) sube el recall")
ejes[0].invert_xaxis(); ejes[0].legend(); ejes[0].grid(alpha=0.3)

ejes[1].plot(tabla["Recall"], tabla["Precision"], marker="o", color="#1428A0")
for _, f in tabla.iterrows():
    ejes[1].annotate(f"{f['Umbral']}", (f["Recall"], f["Precision"]), textcoords="offset points", xytext=(5, 5), fontsize=8)
ejes[1].axhline(real.mean(), ls="--", color="gray", label=f"Azar ({real.mean():.2f})")
ejes[1].set_xlabel("Recall"); ejes[1].set_ylabel("Precision"); ejes[1].set_title("Curva precision-recall (cada punto es un umbral)")
ejes[1].legend(); ejes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Tres cosas para leer en la tabla y los gráficos:**

1. **El recall nunca baja al bajar el umbral.** Su denominador es fijo, y al marcar más casos los aciertos solo pueden mantenerse o subir.
2. **La precision no se mueve de forma pareja.** Pasa de 1,00 (umbral 0,9) a 0,667 (umbral 0,8) y vuelve a subir a 0,75 (umbral 0,75). Depende de si el nuevo caso marcado era un acierto o una falsa alarma. Por eso la curva precision-recall tiene "dientes".
3. **La línea gris es el azar.** Un modelo que marca al azar tiene una precision igual a la proporción de positivos (aquí 0,5). En la curva ROC el azar siempre es la diagonal; en la curva precision-recall depende del desbalance. Por eso esta curva es más informativa cuando la clase positiva es minoritaria.

**Y un matiz:** el intercambio ocurre **dentro de un mismo modelo**. Un modelo mejor puede subir la precision y el recall a la vez; su curva completa queda más arriba y a la derecha.

## 1.4 ¿Por qué F1?

Si hay que elegir un umbral, o comparar modelos con un solo número, necesitamos combinar precision y recall. La idea obvia es promediarlas. El problema es que **el promedio simple premia a los modelos tramposos**.

Miren en la tabla la última fila (umbral 0,05): el modelo marca a **todos** los clientes. El recall es 1,00 sin ningún mérito y la precision es 0,50, igual que el azar. Aun así, su promedio simple es **0,75**, mayor que el del umbral 0,65 (**0,733**), que es un modelo razonable. El F1, en cambio, le da 0,667 al que marca a todos y 0,727 al razonable.

> **Definición. F1:** la media armónica de la precision y el recall.
>
> $$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

La media armónica **se acerca al valor más bajo de los dos**. Para tener un F1 alto, ambas métricas tienen que ser altas: no se puede compensar una mala precision con un recall inflado, ni al revés.

Veámoslo con un caso más extremo: detectar fraudes en 1.000 transacciones, de las cuales 100 son fraude.

In [ ]:
casos = pd.DataFrame({
    "Modelo": ["A: marca solo 1 transacción (y acierta)", "B: marca las 1.000 como fraude", "C: modelo razonable"],
    "Precision": [1.00, 0.10, 0.70],
    "Recall":    [0.01, 1.00, 0.60],
})
casos["Promedio simple"] = (casos["Precision"] + casos["Recall"]) / 2
casos["F1"] = 2 * casos["Precision"] * casos["Recall"] / (casos["Precision"] + casos["Recall"])
casos.round(3)

- **Modelo A:** con el promedio simple saca 0,505, un "aprobado" para un modelo que detecta 1 de 100 fraudes. Su F1 es 0,02.
- **Modelo B:** no hace nada útil (marca todo) y su promedio simple es 0,55, casi igual que A. Su F1 es 0,18.
- **Modelo C:** es el único que sirve, y el F1 lo separa claramente de los otros dos.

### Usar F1 para elegir el umbral

Si no tenemos un criterio de negocio más preciso, un buen punto de partida es **elegir el umbral que maximiza el F1**.

In [ ]:
mejor = tabla.loc[tabla["F1"].idxmax()]
print(f"Umbral con mayor F1: {mejor['Umbral']} | Precision {mejor['Precision']} | Recall {mejor['Recall']} | F1 {mejor['F1']}")

### Cuándo F1 no es suficiente

- **F1 da el mismo peso a precision y recall.** Si un error cuesta más que el otro, se usa **F-beta**: con β = 2, el recall pesa más (útil cuando no detectar es grave, como en fraude o en salud); con β = 0,5, pesa más la precision (útil cuando cada falsa alarma es cara).
- **F1 ignora los verdaderos negativos.** Está pensado para cuando lo que importa es la clase positiva.
- **Si se conocen los costos reales** (cuánto cuesta una llamada de retención y cuánto se pierde con un cliente que se va), lo mejor es elegir el umbral que minimiza el costo total, como hicimos con la tabla de umbrales de la Clase 6.

In [ ]:
from sklearn.metrics import f1_score, fbeta_score

marcados = (prob >= 0.45).astype(int)
print("F1:         ", round(f1_score(real, marcados), 3))
print("F2 (recall pesa más):    ", round(fbeta_score(real, marcados, beta=2), 3))
print("F0,5 (precision pesa más):", round(fbeta_score(real, marcados, beta=0.5), 3))

## Ejercicios de métricas

### Ejercicio 1
Calcula **a mano** (sin usar `metricas_umbral`) los VP, FP y FN del umbral 0,65, y luego la precision, el recall y el F1. Escribe los resultados como números.

In [ ]:
vp_065 = None         # TU CÓDIGO AQUÍ
fp_065 = None
fn_065 = None
precision_065 = None  # puedes escribir la fracción, por ejemplo 3 / 4
recall_065 = None
f1_065 = None

In [ ]:
revisar(lambda: (vp_065, fp_065, fn_065) == (4, 1, 2) and np.isclose(precision_065, 0.8)
        and np.isclose(recall_065, 4 / 6) and np.isclose(f1_065, 2 * 0.8 * (4 / 6) / (0.8 + 4 / 6)))

### Ejercicio 2
Un detector de spam revisa 500 correos, de los cuales 50 son spam. Marca 40 correos como spam y de esos, 30 lo son. Calcula `precision_spam`, `recall_spam` y `f1_spam`.

In [ ]:
precision_spam = None  # TU CÓDIGO AQUÍ
recall_spam = None
f1_spam = None

In [ ]:
revisar(lambda: np.isclose(precision_spam, 30 / 40) and np.isclose(recall_spam, 30 / 50)
        and np.isclose(f1_spam, 2 * 0.75 * 0.6 / 1.35))

---
# Parte 2. ¿Qué es el aprendizaje no supervisado?

Hasta ahora todos los modelos tenían una **variable objetivo** (y): la gama del diamante, si el cliente se va. Eso es **aprendizaje supervisado**: el modelo aprende de ejemplos con la respuesta correcta.

> **Definición. Aprendizaje no supervisado:** el modelo recibe solo las características (X), **sin respuesta correcta**, y busca estructura en los datos por sí mismo.

| | Supervisado | No supervisado |
|---|---|---|
| ¿Hay variable objetivo? | Sí | No |
| Pregunta típica | ¿Este cliente se va? | ¿Qué tipos de clientes tenemos? |
| Cómo se evalúa | Comparando con la respuesta real | No hay respuesta real: con métricas internas y con criterio de negocio |
| Ejemplos | Regresión, árboles, random forest | K-medias, clustering jerárquico, DBSCAN, PCA |

Dos grandes familias:

1. **Clustering (agrupamiento):** formar grupos de observaciones parecidas. Ejemplo: segmentar clientes.
2. **Reducción de dimensionalidad:** resumir muchas columnas en pocas, perdiendo la menor información posible. Ejemplo: PCA.

**Regla que vale para casi todo el capítulo:** estos métodos usan **distancias** entre observaciones, así que **hay que escalar los datos** antes. Si no, la variable con números más grandes domina todo.

---
# Parte 3. K-medias: segmentar clientes

> **Definición. K-medias (k-means):** divide los datos en **k grupos**. Cada grupo tiene un **centroide** (su punto promedio) y cada observación pertenece al grupo del centroide más cercano.

El algoritmo, paso a paso:

1. Elegir k y ubicar k centroides al azar.
2. Asignar cada punto al centroide más cercano.
3. Mover cada centroide al promedio de los puntos de su grupo.
4. Repetir los pasos 2 y 3 hasta que los centroides dejen de moverse.

Lo aplicamos al mismo dataset de la Clase 6, pero ahora **sin mirar `Churn`**: queremos descubrir qué tipos de clientes hay, usando solo la antigüedad (`tenure`) y el cargo mensual (`MonthlyCharges`).

In [ ]:
import os

def cargar_telco():
    try:
        import kagglehub
        carpeta = kagglehub.dataset_download("blastchar/telco-customer-churn")
        return pd.read_csv(os.path.join(carpeta, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))
    except Exception:
        return pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")

clientes = cargar_telco()
datos_seg = clientes[["tenure", "MonthlyCharges"]]
print("Forma:", datos_seg.shape)

plt.figure(figsize=(7, 4.5))
plt.scatter(datos_seg["tenure"], datos_seg["MonthlyCharges"], s=5, alpha=0.3)
plt.xlabel("Antigüedad (meses)"); plt.ylabel("Cargo mensual (USD)"); plt.title("Clientes sin agrupar")
plt.show()

### Escalar y probar varios k

¿Cuántos grupos? K-medias no lo decide solo. Usamos dos herramientas:

> **Inercia:** suma de las distancias al cuadrado de cada punto a su centroide. Siempre baja al aumentar k, así que buscamos el **codo**: el punto donde deja de bajar mucho.
>
> **Coeficiente de silueta:** mide, de -1 a 1, qué tan cerca está cada punto de su grupo comparado con el grupo vecino. **Mientras más alto, mejor separados están los grupos.**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_seg = StandardScaler().fit_transform(datos_seg)

ks = range(2, 9)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_seg)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_seg, km.labels_, sample_size=3000, random_state=0))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].plot(ks, inercias, marker="o"); ejes[0].set_title("Método del codo"); ejes[0].set_xlabel("k"); ejes[0].set_ylabel("Inercia")
ejes[1].plot(ks, siluetas, marker="o", color="#1428A0"); ejes[1].set_title("Coeficiente de silueta"); ejes[1].set_xlabel("k")
plt.tight_layout(); plt.show()
print("Silueta por k:", dict(zip(ks, np.round(siluetas, 3))))

El codo aparece en k = 4 y la silueta también es máxima en k = 4. Las dos herramientas coinciden, así que elegimos **4 segmentos**.

In [ ]:
km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_seg)
clientes["segmento"] = km4.labels_

plt.figure(figsize=(7, 4.5))
sns.scatterplot(data=clientes, x="tenure", y="MonthlyCharges", hue="segmento", palette="tab10", s=8, alpha=0.6)
plt.xlabel("Antigüedad (meses)"); plt.ylabel("Cargo mensual (USD)"); plt.title("Cuatro segmentos encontrados por K-medias")
plt.show()

### Darle sentido a los segmentos

Un clustering solo sirve si se puede interpretar. Describimos cada segmento y, **ahora sí**, miramos la fuga de clientes de cada uno, que el modelo nunca vio.

In [ ]:
perfil = clientes.groupby("segmento").agg(
    clientes=("tenure", "size"),
    antiguedad_media=("tenure", "mean"),
    cargo_medio=("MonthlyCharges", "mean"),
    tasa_fuga=("Churn", lambda s: (s == "Yes").mean()),
).round(2).sort_values("tasa_fuga", ascending=False)
perfil

Los segmentos tienen interpretación de negocio clara:

- **Nuevos con cargo alto:** la fuga más alta, cerca de la mitad. Prioridad para retención.
- **Nuevos con cargo bajo:** fuga intermedia.
- **Antiguos con cargo alto:** clientes valiosos y bastante estables.
- **Antiguos con cargo bajo:** los más fieles, con fuga cercana al 5%.

Un algoritmo **sin respuesta correcta** encontró grupos que se relacionan fuertemente con la fuga. En un proyecto, esto sirve para diseñar estrategias distintas por segmento, o para agregar el segmento como característica a un modelo supervisado.

### Ejercicio 3
Entrena un K-medias con **k = 3** sobre `X_seg` (con `n_init=10` y `random_state=42`) y guarda su coeficiente de silueta en `silueta_k3`. Usa `sample_size=3000, random_state=0` en `silhouette_score`, como arriba. ¿Es mejor o peor que k = 4?

In [ ]:
silueta_k3 = None  # TU CÓDIGO AQUÍ
print(silueta_k3)

In [ ]:
revisar(lambda: np.isclose(silueta_k3, siluetas[1]))

### Ejercicio 4
¿Qué pasa si **no escalamos**? Imaginemos que el cargo mensual estuviera en **pesos chilenos** en vez de dólares. La celda crea `datos_clp` con ese cambio. Entrena un K-medias con k = 4 directamente sobre `datos_clp` (sin escalar) y guarda las etiquetas en `etiquetas_sin_escalar`. Luego grafica el resultado. ¿Qué variable domina la división y por qué?

In [ ]:
datos_clp = datos_seg.assign(MonthlyCharges=datos_seg["MonthlyCharges"] * 950)   # tipo de cambio de ejemplo

etiquetas_sin_escalar = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: np.array_equal(etiquetas_sin_escalar, KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(datos_clp)))

---
# Parte 4. Clustering jerárquico

> **Definición. Clustering jerárquico aglomerativo:** parte con cada observación como un grupo propio y, en cada paso, **une los dos grupos más parecidos**, hasta que todo queda en un solo grupo. El resultado es un árbol llamado **dendrograma**.

Ventaja frente a K-medias: no hay que elegir k de antemano. Se mira el dendrograma y se "corta" a la altura que tenga sentido.

> **Criterio de enlace (linkage):** define la distancia entre dos grupos. El más usado es **Ward**, que une los grupos que menos aumentan la variación interna. Otros: *single* (los puntos más cercanos), *complete* (los más lejanos) y *average* (el promedio).

Usamos el dataset **wine** de scikit-learn: 178 vinos de tres productores, con 13 medidas químicas. Tomamos 40 vinos para que el dendrograma se pueda leer.

In [ ]:
from sklearn.datasets import load_wine
from scipy.cluster.hierarchy import linkage, dendrogram

vinos = load_wine(as_frame=True)
X_vino = StandardScaler().fit_transform(vinos.data)
y_vino = vinos.target          # el productor real: NO se usa para agrupar, solo para comparar al final

rng = np.random.default_rng(0)
idx = rng.choice(len(X_vino), size=40, replace=False)
enlaces = linkage(X_vino[idx], method="ward")

plt.figure(figsize=(12, 4.5))
dendrogram(enlaces, labels=[f"P{p}" for p in y_vino.values[idx]], leaf_font_size=8, color_threshold=12)
plt.axhline(12, ls="--", color="gray")
plt.title("Dendrograma de 40 vinos (la etiqueta indica el productor real)")
plt.ylabel("Distancia de unión"); plt.show()

**Cómo leerlo:** abajo cada vino es un grupo. Hacia arriba se van uniendo. La altura de cada unión indica cuán distintos eran los grupos que se unieron. Si cortamos con la línea gris, quedan **3 grupos**, y cada grupo concentra mayoritariamente a un productor (etiquetas P0, P1 o P2), aunque el algoritmo nunca vio esa información. Hay algunos vinos mezclados: el agrupamiento no es perfecto, y eso es normal.

Con scikit-learn, para todo el dataset:

In [ ]:
from sklearn.cluster import AgglomerativeClustering

jerarquico = AgglomerativeClustering(n_clusters=3, linkage="ward").fit(X_vino)
pd.crosstab(jerarquico.labels_, y_vino, rownames=["Grupo encontrado"], colnames=["Productor real"])

Los números de los grupos son arbitrarios (el "grupo 0" no tiene por qué ser el "productor 0"). Lo que importa es que cada grupo concentre a un productor.

---
# Parte 5. DBSCAN: grupos con forma libre

K-medias supone que los grupos son "bolitas" alrededor de un centroide. Cuando los grupos tienen otras formas, falla.

> **Definición. DBSCAN:** forma grupos con las zonas donde hay **muchos puntos juntos** (alta densidad). Dos parámetros:
>
> - `eps`: el radio del vecindario de cada punto.
> - `min_samples`: cuántos vecinos necesita un punto dentro de ese radio para ser "núcleo" de un grupo.
>
> Los puntos que no quedan en ninguna zona densa se marcan como **ruido** (etiqueta -1). No hay que elegir k.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

X_luna, y_luna = make_moons(n_samples=300, noise=0.07, random_state=0)
X_luna = StandardScaler().fit_transform(X_luna)

etiquetas_km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X_luna)
etiquetas_db = DBSCAN(eps=0.3, min_samples=5).fit_predict(X_luna)

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
ejes[0].scatter(X_luna[:, 0], X_luna[:, 1], c=etiquetas_km, cmap="coolwarm", s=15); ejes[0].set_title("K-medias (k = 2)")
ejes[1].scatter(X_luna[:, 0], X_luna[:, 1], c=etiquetas_db, cmap="coolwarm", s=15); ejes[1].set_title("DBSCAN (eps = 0,3)")
plt.show()
print("Grupos DBSCAN:", sorted(set(etiquetas_db)), "| Puntos de ruido:", (etiquetas_db == -1).sum())

K-medias corta las lunas por la mitad con una línea recta. DBSCAN sigue la forma de cada luna.

### Ejercicio 5
Prueba DBSCAN con `eps=0.1`, `eps=0.2` y `eps=0.5` (con `min_samples=5`). Para cada uno, guarda en el diccionario `n_grupos` cuántos grupos encuentra **sin contar el ruido** (la etiqueta -1). ¿Qué pasa con un eps muy chico y con uno muy grande?

In [ ]:
n_grupos = {}
for eps in [0.1, 0.2, 0.5]:
    pass  # TU CÓDIGO AQUÍ
n_grupos

In [ ]:
_esp = {e: len(set(DBSCAN(eps=e, min_samples=5).fit_predict(X_luna)) - {-1}) for e in [0.1, 0.2, 0.5]}
revisar(lambda: n_grupos == _esp)

---
# Parte 6. PCA: resumir muchas columnas en pocas

> **Definición. Análisis de componentes principales (PCA):** crea nuevas variables, llamadas **componentes**, que son combinaciones de las originales. La primera componente es la dirección en que los datos **varían más**; la segunda, la que varía más entre las perpendiculares a la primera, y así sucesivamente.

Para qué sirve:

1. **Visualizar** datos de muchas dimensiones en 2D.
2. **Reducir** el número de columnas antes de un modelo, cuando hay muchas y están correlacionadas.
3. **Quitar ruido**, quedándose con las componentes que explican más variación.

> **Varianza explicada:** proporción de la variación total de los datos que captura cada componente.

Los vinos tienen 13 medidas químicas: imposible de graficar. Con PCA las resumimos en 2.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(X_vino)
var = pca.explained_variance_ratio_

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
ejes[0].bar(range(1, 14), var, color="#1428A0", label="Por componente")
ejes[0].plot(range(1, 14), var.cumsum(), marker="o", color="orange", label="Acumulada")
ejes[0].set_xlabel("Componente"); ejes[0].set_ylabel("Varianza explicada"); ejes[0].set_title("¿Cuánta información guarda cada componente?")
ejes[0].legend()

Z = PCA(n_components=2).fit_transform(X_vino)
sc = ejes[1].scatter(Z[:, 0], Z[:, 1], c=y_vino, cmap="viridis", s=20)
ejes[1].set_xlabel("Componente 1"); ejes[1].set_ylabel("Componente 2"); ejes[1].set_title("13 medidas resumidas en 2 (color: productor real)")
plt.colorbar(sc, ax=ejes[1]); plt.tight_layout(); plt.show()

print(f"Las 2 primeras componentes explican el {var[:2].sum():.1%} de la variación total")

Con solo 2 de 13 dimensiones, que guardan algo más de la mitad de la variación, los tres productores se ven claramente separados.

### PCA + K-medias

¿Encuentra K-medias los tres productores sin conocerlos? Lo medimos con el **índice de Rand ajustado (ARI)**, que compara dos agrupaciones: 1 es coincidencia perfecta y 0 es lo mismo que el azar. Es una forma de evaluar un clustering **cuando, excepcionalmente, conocemos la respuesta**, como aquí.

In [ ]:
from sklearn.metrics import adjusted_rand_score

km_13 = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_vino)
km_2 = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(Z)
print(f"K-medias con las 13 medidas: ARI = {adjusted_rand_score(y_vino, km_13):.3f}")
print(f"K-medias con 2 componentes:   ARI = {adjusted_rand_score(y_vino, km_2):.3f}")

Con 2 componentes se obtiene prácticamente el mismo resultado que con las 13 columnas. PCA comprimió los datos sin perder lo importante para separar los grupos.

### Ejercicio 6
¿Cuántas componentes se necesitan para explicar **al menos el 80%** de la variación de los vinos? Usa `var.cumsum()` y guarda la respuesta en `n_componentes_80`.

In [ ]:
n_componentes_80 = None  # TU CÓDIGO AQUÍ
print(n_componentes_80)

In [ ]:
revisar(lambda: n_componentes_80 == int(np.argmax(var.cumsum() >= 0.8) + 1))

---
# Cierre

## Resumen de métricas

| Métrica | Fórmula | Pregunta |
|---|---|---|
| Precision | VP / (VP + FP) | De lo que marqué, ¿cuánto era correcto? |
| Recall | VP / (VP + FN) | De lo real, ¿cuánto encontré? |
| F1 | 2·P·R / (P + R) | ¿Están ambas altas a la vez? |

El intercambio entre precision y recall lo produce el **umbral**. F1 sirve para elegir el umbral o comparar modelos cuando ambas importan por igual.

## Resumen de métodos no supervisados

| Método | Qué hace | Hay que elegir | Fortaleza | Debilidad |
|---|---|---|---|---|
| K-medias | Grupos alrededor de centroides | k | Rápido y fácil de interpretar | Supone grupos redondos y de tamaño parecido |
| Jerárquico | Une grupos paso a paso | Dónde cortar el dendrograma | Muestra la estructura completa | Lento con muchos datos |
| DBSCAN | Grupos por densidad | eps y min_samples | Formas libres, detecta ruido | Sensible a eps; difícil con densidades distintas |
| PCA | Resume columnas en componentes | Número de componentes | Visualizar y comprimir | Las componentes son difíciles de interpretar |

**Para el capstone:** un proyecto supervisado puede usar estos métodos como apoyo, por ejemplo segmentar clientes antes de predecir, o usar PCA para visualizar los datos en el EDA.

Guarden el notebook en `ejercicios_estudiantes` como `clase07_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1
Con umbral 0,65 el modelo marca a C01, C02, C03, C04 y C05. Aciertos: C01, C02, C04 y C05 (VP = 4). Falsa alarma: C03 (FP = 1). No detectados: C07 y C10 (FN = 2).

In [ ]:
vp_065, fp_065, fn_065 = 4, 1, 2
precision_065 = 4 / (4 + 1)                    # 0,8
recall_065 = 4 / (4 + 2)                       # 0,667
f1_065 = 2 * precision_065 * recall_065 / (precision_065 + recall_065)   # 0,727
print(round(precision_065, 3), round(recall_065, 3), round(f1_065, 3))

### Ejercicio 2
Marcó 40 y acertó 30: VP = 30, FP = 10. Había 50 spam y encontró 30: FN = 20.

In [ ]:
precision_spam = 30 / 40     # 0,75
recall_spam = 30 / 50        # 0,60
f1_spam = 2 * precision_spam * recall_spam / (precision_spam + recall_spam)   # 0,667
print(precision_spam, recall_spam, round(f1_spam, 3))

### Ejercicio 3

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_seg)
silueta_k3 = silhouette_score(X_seg, km3.labels_, sample_size=3000, random_state=0)
print(round(silueta_k3, 3), "(peor que k = 4)")

### Ejercicio 4
En pesos, el cargo mensual va de unos 17.000 a 113.000, mientras la antigüedad va de 0 a 72. En la distancia, una diferencia de 1.000 pesos pesa lo mismo que 1.000 meses, así que el cargo domina todo: los grupos quedan como franjas horizontales y la antigüedad no influye. La misma información, con otra unidad, da un clustering completamente distinto. Por eso hay que escalar.

In [ ]:
etiquetas_sin_escalar = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(datos_clp)
plt.figure(figsize=(7, 4.5))
plt.scatter(datos_clp["tenure"], datos_clp["MonthlyCharges"], c=etiquetas_sin_escalar, cmap="tab10", s=5)
plt.xlabel("Antigüedad (meses)"); plt.ylabel("Cargo mensual (CLP)"); plt.title("K-medias sin escalar: domina el cargo en pesos")
plt.show()

### Ejercicio 5
Con eps muy chico, casi todos los puntos quedan aislados: muchos grupos pequeños y mucho ruido. Con eps muy grande, todo queda en un solo grupo.

In [ ]:
n_grupos = {}
for eps in [0.1, 0.2, 0.5]:
    etiquetas = DBSCAN(eps=eps, min_samples=5).fit_predict(X_luna)
    n_grupos[eps] = len(set(etiquetas) - {-1})
n_grupos

### Ejercicio 6

In [ ]:
n_componentes_80 = int(np.argmax(var.cumsum() >= 0.8) + 1)   # argmax devuelve la primera posición que cumple
print(n_componentes_80, "componentes explican", round(var.cumsum()[n_componentes_80 - 1], 3))